In [6]:
#!/usr/bin/env python3
"""Finite checks for rm19_minimal.tex.

Install: python -m pip install numpy numba
Run:     python verify_rm19_colab.py

The catalog is included below. No downloads, C++ compiler, previous logs, or
other input files are used. NumPy supplies arrays; Numba compiles the integer
loops. The default run audits the catalog and performs both full enumerations.

In a notebook, call run_verification(workers=None, output="rm19_verification.json").
The companion Colab notebook includes a dependency-installation cell. A CPU
runtime is sufficient. Every invocation starts a fresh verification.

The program checks the finite part of the proof. It does not prove the analytic
lemmas or rederive the published count of 349 cubic-part orbits.
"""
from __future__ import annotations

import argparse
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import os
from pathlib import Path
import platform
import re
import sys
from time import perf_counter

try:
    import numpy as np
    import numba
    from numba import njit, literal_unroll
except ImportError as exc:
    raise SystemExit("Install the dependencies: python -m pip install numpy numba") from exc

N = 512
X = np.arange(N, dtype=np.int64)
QUADRATICS = [m for m in range(N) if m.bit_count() == 2]
Q_INDEX = {m: j for j, m in enumerate(QUADRATICS)}
H = np.array([[1 - 2 * ((a & x).bit_count() % 2)
               for x in range(N)] for a in range(N)], dtype=np.int64)
H_TEN = H[[0] + [1 << i for i in range(9)]]
EXPECTED_DIMS = {18: 142, 19: 20, 20: 18, 21: 6, 22: 5, 24: 1, 27: 1}
EXPECTED_TOTALS = (251133952, 90126659, 240)
CATALOG_SHA256 = "a3a284737f94ef8c0a4cb198dfcb2328a90349517517decf3e74dfccbce42922"
WITNESS = (
    "68B7EF2DA03B0D3EA00DB6A96DD99AEAFDB9C842B6D5DC8C4526CE0DD29020DB"
    "B75FE3314568344E73688FF0CB2482E065231869E1AA4583765CC491F8A8DB12"
)


def require(condition, message):
    # Unlike assert, this check remains active under python -O.
    if not condition:
        raise RuntimeError(message)


def parse_catalog():
    require(hashlib.sha256(CATALOG.encode("ascii")).hexdigest() == CATALOG_SHA256,
            "The embedded catalog has changed.")
    forms = []
    for line in CATALOG.splitlines():
        match = re.fullmatch(r"\s*(\d+) h=([0-9a-f]+)\s*(.*?)\s*", line)
        require(match is not None, "Malformed catalog record.")
        terms = match[3].split("+")[1:] if match[3] else []
        require(all(len(t) == 3 and len(set(t)) == 3 and
                    all(ch in "123456789" for ch in t) for t in terms),
                "A catalog monomial is not squarefree cubic.")
        mon = [sum(1 << (int(ch) - 1) for ch in t) for t in terms]
        require(len(mon) == len(set(mon)), "Repeated monomial in catalog.")
        forms.append((int(match[1]), mon))
    require([i for i, _ in forms] == list(range(1, 350)), "Wrong catalog IDs.")
    return forms


def truth(mon, q=0):
    """Truth table of c+q; bit j of q selects QUADRATICS[j]."""
    values = np.zeros(N, dtype=np.int64)
    for m in mon:
        values ^= (X & m) == m
    for j, m in enumerate(QUADRATICS):
        if (q >> j) & 1:
            values ^= (X & m) == m
    return values


def walsh(values):
    """Separate vectorized integer transform, used in preparation."""
    out = np.array(values, dtype=np.int64, copy=True)
    width = 1
    while width < len(out):
        blocks = out.reshape(-1, 2 * width)
        left, right = blocks[:, :width].copy(), blocks[:, width:].copy()
        blocks[:, :width], blocks[:, width:] = left + right, left - right
        width *= 2
    return out


@njit(inline="always")
def walsh_up(w):
    for width in literal_unroll((1, 2, 4, 8, 16, 32, 64, 128, 256)):
        for block in range(0, 512, 2 * width):
            for j in range(width):
                u, v = w[block + j], w[block + j + width]
                w[block + j], w[block + j + width] = u + v, u - v


@njit(inline="always")
def walsh_down(w):
    width = 256
    while width:
        for block in range(0, 512, 2 * width):
            for j in range(width):
                u, v = w[block + j], w[block + j + width]
                w[block + j], w[block + j + width] = u + v, u - v
        width //= 2


@njit
def walsh_columns(a):
    """Transform each column, with points indexing rows."""
    out = a.copy()
    width = 1
    while width < out.shape[0]:
        for block in range(0, out.shape[0], 2 * width):
            for j in range(width):
                for k in range(out.shape[1]):
                    u, v = out[block + j, k], out[block + j + width, k]
                    out[block + j, k], out[block + j + width, k] = u + v, u - v
        width *= 2
    return out


def self_tests(forms):
    bits = np.array([int(b) for h in WITNESS for b in f"{int(h, 16):04b}"],
                    dtype=np.int64)
    require(len(bits) == N, "Wrong witness length.")
    spectrum = H @ (1 - 2 * bits)  # All 512 direct character sums.
    require(int(np.max(np.abs(spectrum))) == 28, "Witness is not NL 242.")
    tests = [1 - 2 * bits, np.ones(N, dtype=np.int64)]
    tests += [1 - 2 * truth(mon) for _, mon in forms[::17]]
    for phase in tests:
        direct = H @ phase
        up, down = phase.astype(np.int16), phase.astype(np.int32)
        walsh_up(up)
        walsh_down(down)
        require(np.array_equal(walsh(phase), direct) and
                np.array_equal(up, direct) and np.array_equal(down, direct),
                "Walsh transform disagrees with direct sums.")
    # All possible nine-variable Walsh values, with denominators cleared.
    maxima = []
    for residue, sign in [(8, -1), (24, 1)]:
        corrections = []
        for w in range(-512, 513):
            if w % 32 == residue:
                numerator = 3 * w * w + sign * 16 * w + 960
                require(numerator >= 128 * abs(w), "Cubic majorant failed.")
                corrections.append(384 * w - numerator)
        maxima.append(max(corrections) // 128)
    require(maxima == [96, 80], "Wrong lattice correction maxima.")
    for t in range(-32, 33):
        if t % 2 == 0:
            require(2 * abs(t) <= t*t and 6*t - t*t <= 8,
                    "Even-parity majorant failed.")
        else:
            require(4 * abs(t) <= t*t + 3 and 12*t - t*t - 3 <= 32,
                    "Odd-parity majorant failed.")
        require(2 * abs(t) <= t*t + (t % 2) and
                6*t - t*t - (t % 2) <= 8, "Mixed-parity majorant failed.")
    return {"witness_walsh_max": 28, "witness_nonlinearity": 242,
            "direct_transform_tests": len(tests), "majorants_checked": True}


def reduce_vector(v, basis, high=True):
    for pivot in sorted(basis, reverse=high):
        if (v >> pivot) & 1:
            v ^= basis[pivot]
    return v


def add_vector(v, basis, high=True):
    v = reduce_vector(v, basis, high)
    if not v:
        return False
    pivot = v.bit_length() - 1 if high else (v & -v).bit_length() - 1
    basis[pivot] = v
    return True


def rank(vectors):
    basis = {}
    for v in vectors:
        add_vector(v, basis)
    return len(basis)


def image(q, columns):
    out = 0
    for j, col in enumerate(columns):
        if (q >> j) & 1:
            out ^= col
    return out


def solve_rows(columns, target):
    """First solution: row reduction of the augmented 9 by 36 system."""
    rows = [sum(((c >> i) & 1) << j for j, c in enumerate(columns)) |
            (((target >> i) & 1) << 36) for i in range(9)]
    pivots = []
    for j in range(36):
        k = len(pivots)
        pivot = next((i for i in range(k, 9) if (rows[i] >> j) & 1), None)
        if pivot is None:
            continue
        rows[k], rows[pivot] = rows[pivot], rows[k]
        for i in range(9):
            if i != k and ((rows[i] >> j) & 1):
                rows[i] ^= rows[k]
        pivots.append(j)
    if any((r & ((1 << 36) - 1)) == 0 and (r >> 36) for r in rows):
        return None
    offset = sum(((rows[i] >> 36) & 1) << j for i, j in enumerate(pivots))
    kernel = [(1 << j) | sum(((rows[i] >> j) & 1) << p
                            for i, p in enumerate(pivots))
              for j in range(36) if j not in pivots]
    return offset, kernel


def solve_columns(columns, target):
    """Second solution: eliminate columns, retaining their representations."""
    basis, kernel = {}, []

    def reduce_pair(v, rep):
        for p in sorted(basis):
            if (v >> p) & 1:
                w, r = basis[p]
                v, rep = v ^ w, rep ^ r
        return v, rep

    for j in reversed(range(36)):
        v, rep = reduce_pair(columns[j], 1 << j)
        if v:
            basis[(v & -v).bit_length() - 1] = (v, rep)
        else:
            kernel.append(rep)
    v, offset = reduce_pair(target, 0)
    if v:
        return None
    # A different point of the same affine solution space.
    if kernel:
        offset ^= kernel[-1]
    return offset, list(reversed(kernel))


def prepare(forms, second=False):
    cases, counts = [], Counter()
    for number, mon in forms:
        base_phase = 1 - 2 * truth(mon)
        base_w0 = int(H_TEN[0] @ base_phase) if second else int(walsh(base_phase)[0])
        require(base_w0 % 8 == 0, "Cubic divisibility failed.")
        if base_w0 % 16 == 8:
            counts["odd_residue"] += 1
            continue
        derivatives = []
        for i in range(9):
            d = 0
            for m in mon:
                if (m >> i) & 1:
                    d ^= 1 << Q_INDEX[m ^ (1 << i)]
            derivatives.append(d)
        if rank(derivatives) < 9:
            counts["degenerate"] += 1
            continue

        def phi(q):
            phase = 1 - 2 * truth(mon, q)
            if second:
                w = H_TEN @ phase
                differences = w[1:] - w[0]
            else:
                w = walsh(phase)
                differences = w[[1 << i for i in range(9)]] - w[0]
            require(np.all(w % 16 == 0), "Wrong cubic residue.")
            return sum(int((d // 16) % 2) << i for i, d in enumerate(differences))

        target = phi(0)
        columns = [phi(1 << j) ^ target for j in range(36)]
        solution = (solve_columns if second else solve_rows)(columns, target)
        if solution is None:
            counts["inconsistent"] += 1
            continue
        offset, kernel = solution
        basis, complement = {}, []
        for d in (list(reversed(derivatives)) if second else derivatives):
            require(add_vector(d, basis, not second), "Dependent derivative basis.")
        for v in kernel:
            if add_vector(v, basis, not second):
                complement.append(v)
        require(len(kernel) == 36 - rank(columns), "Wrong kernel dimension.")
        require(len(complement) + 9 == len(kernel), "Wrong quotient dimension.")
        require(rank(derivatives + complement) == len(kernel), "Wrong complement.")
        require(image(offset, columns) == target, "Wrong affine offset.")
        for v in derivatives + complement:
            require(image(v, columns) == 0 and phi(offset ^ v) == 0,
                    "A representative fails the affine equations.")
        require(phi(offset) == 0, "Offset fails the affine equations.")
        cases.append((number, mon, offset, complement))
        counts[len(complement)] += 1
    expected = Counter(EXPECTED_DIMS)
    expected.update({"odd_residue": 124, "degenerate": 32})
    require(counts == expected, "Preparation counts differ from the manuscript.")
    require(sum(1 << len(b) for _, _, _, b in cases) == EXPECTED_TOTALS[0],
            "Wrong size of the search space.")
    return cases


@njit
def contractions(mon):
    """Ranks and row-space membership for all T(u,.,.)."""
    partial = np.zeros((9, 9), dtype=np.int32)
    for m in mon:
        for k in range(9):
            if (m >> k) & 1:
                rest = m ^ (1 << k)
                for i in range(9):
                    if (rest >> i) & 1:
                        partial[k, i] ^= rest ^ (1 << i)
    ranks = np.zeros(512, dtype=np.int32)
    inside = np.zeros((512, 512), dtype=np.uint8)
    for u in range(512):
        rows = np.zeros(9, dtype=np.int32)
        for k in range(9):
            if (u >> k) & 1:
                for i in range(9):
                    rows[i] ^= partial[k, i]
        basis = np.zeros(9, dtype=np.int32)
        generators = np.zeros(9, dtype=np.int32)
        r = 0
        for row in rows:
            for i in range(8, -1, -1):
                if (row >> i) & 1:
                    if basis[i]:
                        row ^= basis[i]
                    else:
                        basis[i] = row
                        generators[r] = row
                        r += 1
                        break
        if r % 2:
            raise RuntimeError("An alternating matrix has odd rank.")
        ranks[u] = r
        span = np.zeros(512, dtype=np.int32)
        size = 1
        for j in range(r):
            for i in range(size):
                span[size + i] = span[i] ^ generators[j]
            size *= 2
        for i in range(size):
            inside[u, span[i]] = 1
    return ranks, inside


def hyperplanes():
    points = []
    for a in range(1, N):
        pivot = (a & -a).bit_length() - 1
        span = [0]
        for k in range(9):
            if k != pivot:
                g = (1 << k) ^ (((a >> k) & 1) << pivot)
                span += [v ^ g for v in span]
        require(len(span) == 256 and all((a & u).bit_count() % 2 == 0 for u in span),
                "Wrong hyperplane basis.")
        points.append(span)
    return np.array(points, dtype=np.int32)


@njit
def restriction_spectra(ranks, inside, points):
    spectra = np.zeros((511, 256, 4), dtype=np.int32)
    for a in range(1, 512):
        for i in range(256):
            u = points[a - 1, i]
            r = ranks[u] - 2 * int(inside[u, a])
            if r < 0 or r > 6 or r % 2:
                raise RuntimeError("Invalid restricted contraction rank.")
            spectra[a - 1, i, r // 2] = 1
        spectra[a - 1] = walsh_columns(spectra[a - 1])
    return spectra


def sorted_rows(a):
    # Exact integer rows, not a digest: dictionary collisions cannot merge keys.
    order = np.lexsort(a[:, ::-1].T)
    return np.ascontiguousarray(a[order], dtype="<i4").tobytes()


def audit_catalog(forms):
    points, labels, saved = hyperplanes(), {}, []
    for number, mon in forms:
        ranks, inside = contractions(np.array(mon, dtype=np.int32))
        restrictions = restriction_spectra(ranks, inside, points)
        colors = []
        for s in restrictions:
            key = sorted_rows(s)
            if key not in labels:
                labels[key] = len(labels)
            colors.append(labels[key])
        psi = (ranks[:, None] == 2 * np.arange(5)).astype(np.int32)
        saved.append((ranks, walsh_columns(psi), np.array(colors, dtype=np.int32)))
        if number % 50 == 0:
            print(f"Catalog audit: {number}/349", flush=True)
    # Use the same complete list of restriction labels for every catalog form.
    signatures = set()
    for ranks, psi_hat, colors in saved:
        first = np.column_stack((colors, psi_hat[1:]))
        phi = np.zeros((N, len(labels)), dtype=np.int32)
        phi[np.arange(1, N), colors] = 1
        second = np.column_stack((ranks, walsh_columns(phi)))
        signatures.add((sorted_rows(first), sorted_rows(second)))
    require(len(labels) == 32 and len(signatures) == 349,
            "The catalog audit did not distinguish all 349 forms.")
    return {"distinct_classes": len(signatures), "restriction_labels": len(labels)}


@njit(nogil=True)
def enumerate_signs(base, toggles, begin, end):
    """First enumeration: signs, ascending butterflies, 16-bit arrays."""
    phase = base.copy()
    gray = begin ^ (begin >> 1)
    for j in range(toggles.shape[0]):
        if (gray >> j) & 1:
            for x in range(512):
                phase[x] *= toggles[j, x]
    w = np.empty(512, dtype=np.int16)
    f = np.empty(512, dtype=np.int16)
    good, best = 0, 0
    for it in range(begin, end):
        if it != begin:
            j, z = 0, it
            while z % 2 == 0:
                z //= 2
                j += 1
            for x in range(512):
                phase[x] *= toggles[j, x]
        total = 0
        for x in range(512):
            total += phase[x]
        if total % 32 != 16:
            continue
        for x in range(512):
            w[x] = phase[x]
        walsh_up(w)
        valid = True
        for x in range(512):
            if abs(w[x]) != 16 and abs(w[x]) != 48:
                valid = False
                break
        if not valid:
            continue
        good += 1
        for x in range(512):
            f[x] = 1 if w[x] < 0 else -1
        walsh_up(f)
        maximum = 0
        for x in range(512):
            maximum = max(maximum, abs(f[x]))
        best = max(best, 256 - maximum // 2)
        if maximum <= 24:
            raise RuntimeError("Counterexample: negative Walsh sign has NL at least 244.")
    return end - begin, good, best


@njit(nogil=True)
def enumerate_bits(base, toggles, begin, end):
    """Second enumeration: truth bits, descending butterflies, 32-bit arrays."""
    bits = base.copy()
    gray = begin ^ (begin >> 1)
    for j in range(toggles.shape[0]):
        if (gray >> j) & 1:
            for x in range(512):
                bits[x] ^= toggles[j, x]
    w = np.empty(512, dtype=np.int32)
    f = np.empty(512, dtype=np.int32)
    good, best = 0, 0
    for it in range(begin, end):
        if it != begin:
            j, z = 0, it
            while (z & 1) == 0:
                z >>= 1
                j += 1
            for x in range(512):
                bits[x] ^= toggles[j, x]
        weight = 0
        for x in range(512):
            weight += bits[x]
        if (512 - 2 * weight) % 32 != 16:
            continue
        for x in range(512):
            w[x] = 1 - 2 * int(bits[x])
        walsh_down(w)
        maximum = 0
        for x in range(512):
            if w[x] % 32 != 16:
                raise RuntimeError("A prepared representative has the wrong residue.")
            maximum = max(maximum, abs(w[x]))
        if maximum > 48:
            continue
        good += 1
        for x in range(512):
            f[x] = -1 if w[x] > 0 else 1
        walsh_down(f)
        maximum = 0
        for x in range(512):
            maximum = max(maximum, abs(f[x]))
        best = max(best, 256 - maximum // 2)
        if maximum <= 24:
            raise RuntimeError("Counterexample: negative Walsh sign has NL at least 244.")
    return end - begin, good, best


def available_workers():
    """Use at most four CPUs actually available to this process."""
    try:
        count = len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        count = os.cpu_count() or 1
    return max(1, min(4, count))


def write_report(path, report):
    """Replace the JSON in one operation, rather than leave a half-written file."""
    path = Path(path).expanduser()
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def enumerate_all(cases, second, workers, on_class=None):
    records = {}
    function = enumerate_bits if second else enumerate_signs
    run = 1 + int(second)
    started = last_update = perf_counter()
    cumulative = 0
    print(f"Run {run}: compiling integer loops on first use.", flush=True)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for class_index, (number, mon, offset, basis) in enumerate(cases, 1):
            if second:
                base = truth(mon, offset).astype(np.uint8)
                toggles = np.array([truth([], q) for q in basis], dtype=np.uint8)
            else:
                base = (1 - 2 * truth(mon, offset)).astype(np.int16)
                toggles = np.array([1 - 2 * truth([], q) for q in basis], dtype=np.int16)
            size = 1 << len(basis)
            # Compile before using threads. This empty interval skips no tests.
            function(base, toggles, 0, 0)
            # Each interval restarts at its own Gray-code position. Smaller
            # intervals keep progress visible and make stopping more responsive.
            step = 1 << 18
            jobs = {
                pool.submit(function, base, toggles, begin, min(begin + step, size)):
                (begin, min(begin + step, size))
                for begin in range(0, size, step)
            }
            tested = good = best = 0
            try:
                for job in as_completed(jobs):
                    begin, end = jobs[job]
                    n, g, b = map(int, job.result())
                    require(n == end - begin, "An enumeration interval was incomplete.")
                    tested += n
                    good += g
                    best = max(best, b)
                    now = perf_counter()
                    if now - last_update >= 5 and tested < size:
                        percentage = 100 * (cumulative + tested) / EXPECTED_TOTALS[0]
                        print(f"Run {run}, class {number}: {tested:,}/{size:,}; "
                              f"{percentage:.1f}% of this run; "
                              f"{now - started:.0f} s elapsed", flush=True)
                        last_update = now
            except BaseException:
                # Cancel queued tasks. Already-running intervals are allowed to
                # finish before control returns to the notebook.
                for job in jobs:
                    job.cancel()
                raise
            require(tested == size, "An enumeration interval was omitted.")
            records[number] = [len(basis), tested, good, best]
            cumulative += tested
            if on_class is not None:
                on_class(number, records[number])
            print(f"Run {run}, class {number} ({class_index}/{len(cases)}): "
                  f"{tested:,} tested, {good:,} retained, best {best}", flush=True)
    totals = (sum(r[1] for r in records.values()), sum(r[2] for r in records.values()),
              max(r[3] for r in records.values()))
    require(totals == EXPECTED_TOTALS, "Enumeration totals differ from the manuscript.")
    return records, perf_counter() - started


def run_verification(workers=None, output="rm19_verification.json"):
    """Run every finite check and return the report. No command-line parsing.

    The JSON is updated after each completed class. It is a progress record,
    not a resumable checkpoint. PASS is written only after both full runs agree.
    """
    workers = available_workers() if workers is None else workers
    require(isinstance(workers, int) and not isinstance(workers, bool) and workers >= 1,
            "workers must be a positive integer or None.")
    path = Path(output).expanduser()
    started = perf_counter()
    report = {
        "status": "RUNNING", "stage": "initialization",
        "catalog_sha256": CATALOG_SHA256,
        "environment": {"python": platform.python_version(), "numpy": np.__version__,
                        "numba": numba.__version__, "workers": workers},
        "scope": "Finite checks only; analytic proof and published orbit count are separate.",
        "runs": {"1": {"complete": False, "per_class": {}},
                 "2": {"complete": False, "per_class": {}}},
        "per_class_agreement": False
    }

    def save(stage=None):
        if stage is not None:
            report["stage"] = stage
        report["total_seconds"] = perf_counter() - started
        write_report(path, report)

    def record_class(run):
        def record(number, values):
            report["runs"][str(run)]["per_class"][number] = values
            save()
        return record

    # Clear an earlier PASS before starting any new work.
    save()
    print(f"CPU verification with {workers} worker(s). Results: {path}", flush=True)
    try:
        require(not numba.config.DISABLE_JIT,
                "Numba JIT is disabled; remove NUMBA_DISABLE_JIT and restart the runtime.")
        save("witness_and_arithmetic")
        print("1/5: catalog checksum, witness, transforms, and integer inequalities.\n"
              "The first call also compiles the Numba functions.", flush=True)
        forms = parse_catalog()
        report["elementary_checks"] = self_tests(forms)
        print("Witness and arithmetic checks passed.", flush=True)

        save("preparation")
        print("2/5: preparing both sets of quadratic representatives.", flush=True)
        cases1 = prepare(forms)
        print("First preparation complete.", flush=True)
        cases2 = prepare(forms, second=True)
        require([(i, len(b)) for i, _, _, b in cases1] ==
                [(i, len(b)) for i, _, _, b in cases2], "Preparations disagree.")
        report["quotient_dimensions"] = dict(Counter(len(b) for _, _, _, b in cases1))
        print("Both preparations: 193 classes, 251,133,952 representatives each.", flush=True)

        save("catalog_audit")
        print("3/5: auditing all 349 cubic forms.", flush=True)
        report["catalog_audit"] = audit_catalog(forms)
        print("Catalog: 349 distinct invariants, 32 restriction labels.", flush=True)

        save("enumeration_1")
        print("4/5: first complete enumeration.", flush=True)
        records1, seconds1 = enumerate_all(cases1, False, workers, record_class(1))
        report["runs"]["1"]["complete"] = True
        report["runs"]["1"]["seconds"] = seconds1

        save("enumeration_2")
        print("5/5: second complete enumeration.", flush=True)
        records2, seconds2 = enumerate_all(cases2, True, workers, record_class(2))
        report["runs"]["2"]["complete"] = True
        report["runs"]["2"]["seconds"] = seconds2
        require(records1 == records2, "The two enumerations disagree on a cubic class.")
        report.update({
            "status": "PASS", "per_class_agreement": True,
            "tested_per_run": sum(r[1] for r in records1.values()),
            "retained_per_run": sum(r[2] for r in records1.values()),
            "maximum_sign_nonlinearity": max(r[3] for r in records1.values()),
            "per_class": records1, "enumeration_seconds": [seconds1, seconds2]
        })
        save("complete")
    except BaseException as exc:
        report["status"] = "INTERRUPTED" if isinstance(exc, KeyboardInterrupt) else "FAILED"
        report["error"] = f"{type(exc).__name__}: {exc}"
        try:
            save()
        except OSError as write_error:
            print(f"Could not update the progress report: {write_error}", flush=True)
        print(f"{report['status']}: no completed verification was certified.\n"
              "Rerunning starts from the beginning.", flush=True)
        raise

    print("PASS: all finite checks reproduced; both runs agree on every class.")
    print(f"Each run: {report['tested_per_run']:,} tested, "
          f"{report['retained_per_run']:,} retained, "
          f"maximum sign nonlinearity {report['maximum_sign_nonlinearity']}.")
    print(f"Elapsed: {report['total_seconds']:.1f} seconds. Results: {path}")
    return report


def main(argv=None):
    # Pasting this file into a notebook must not parse the kernel's -f argument.
    # Explicit main([...]) calls and ordinary command-line arguments still work.
    if argv is None and "ipykernel" in sys.modules:
        if ("__file__" not in globals() or
                Path(sys.argv[0]).name in {"ipykernel_launcher.py", "colab_kernel_launcher.py"}):
            argv = []
    parser = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--workers", type=int, default=None,
                        help="CPU threads (default: available CPUs, capped at 4)")
    parser.add_argument("--output", default="rm19_verification.json", help="result JSON file")
    args = parser.parse_args(argv)
    return run_verification(args.workers, args.output)


# Mathematical records from P. Langevin's author-hosted cubic catalog:
# https://langevin.univ-tln.fr/project/cubics/classification.txt
# +ijk means x_i*x_j*x_k. Only this polynomial column is used; h= is ignored.
# These are data, not precomputed search results. All checks above are rerun.
CATALOG = r"""  1 h=8188104ce502c8094812c  +124+125+345+126+146+247+347+167+467+128+138+168+268+378+129+349+369+179+479+189+389
  2 h=0a3dfd278b0c2669c9315  +124+125+145+345+236+346+356+237+347+157+257+367+467+128+138+248+348+158+358+168+368+278+378+478+129+149+259+459+169+269+369+469+479+579+679+289+689
  3 h=0bf80cdf933998684397f  +123+124+125+135+145+126+246+156+256+356+127+237+247+347+257+357+367+567+128+138+238+348+258+358+168+268+368+568+178+478+578+129+139+349+269+569+179+379+479+289+389+489+589
  4 h=7586bff46fbabb3345ed8  +124+134+234+125+145+345+126+146+356+456+127+137+257+357+567+128+238+348+358+168+268+468+378+139+149+159+359+459+369+579+289+389+789
  5 h=8100001000139d0408310  +234+135+136+236+246+127+237+147+247+257+167+238+248+358+569+789
  6 h=8100001400139f0448711  +123+234+135+235+136+236+246+156+356+127+137+237+147+247+257+167+238+248+358+569+789
  7 h=8100201000139d0408293  +235+136+236+347+567+128+148+248+258+458+129+149+159+359+169+189+589
  8 h=8100201000129100083b2  +124+134+234+135+136+246+127+147+257+457+128+238+248+358+569+789
  9 h=8100201000129400083b2  +124+134+135+136+236+246+127+147+257+457+128+238+248+358+569+789
 10 h=81000094211bce050c432  +123+125+145+246+247+257+457+267+467+138+238+248+168+568+178+149+249+259+569+479+679+389+589
 11 h=81000094211bce050c433  +123+145+246+156+256+456+147+257+167+467+248+158+258+168+568+178+149+159+169+179+479+679+389+589
 12 h=81000094211b4e050cc22  +124+235+245+345+236+146+256+356+137+237+147+247+257+167+128+138+248+158+358+268+368+569+789
 13 h=81000094211b4e050cc32  +124+134+125+145+126+136+236+146+246+137+237+247+157+458+678+139+239+359+169+369+569+179+479
 14 h=81000094011bda0108435  +123+125+235+245+246+256+356+127+137+237+147+247+257+128+238+248+158+358+268+368+569+789
 15 h=81020084211bdf050c638  +123+134+234+146+347+567+148+158+458+368+129+139+149+349+159+359+169+469+489+589+689
 16 h=81020084211bdf054c63a  +124+234+235+245+345+126+136+236+146+246+156+256+356+127+137+237+147+247+257+167+128+238+248+158+268+368+468+569+789
 17 h=81020084219bdf05cce33  +123+124+234+235+245+345+136+236+146+246+156+256+356+127+137+237+147+247+157+257+167+267+128+138+238+248+158+268+368+468+569+789
 18 h=81021004e51ac909486b2  +124+134+234+245+345+136+246+346+156+256+356+456+127+137+237+147+347+257+128+248+158+168+268+468+278+569+789
 19 h=81801004e51ac909481aa  +124+134+234+135+245+345+126+246+346+156+256+356+456+237+147+347+257+128+248+158+168+268+568+278+569+789
 20 h=81801004e51a4909485b2  +124+134+234+135+245+345+346+156+256+356+456+127+137+237+147+347+257+128+248+158+168+268+568+278+569+789
 21 h=81801004e51a4909487b3  +123+124+134+234+135+245+345+136+346+156+256+356+456+127+137+237+147+347+257+128+248+158+168+268+568+278+569+789
 22 h=8180100ce51ac909481aa  +124+134+234+135+245+345+126+246+346+156+256+356+456+237+147+347+257+357+128+248+158+168+268+568+278+569+789
 23 h=8188100ce512cb09481b8  +134+234+135+235+245+345+126+246+346+156+356+456+127+237+147+347+257+357+128+248+158+168+268+568+278+478+569+789
 24 h=8100525720189a29420c2  +124+134+235+145+345+246+156+256+356+456+237+257+367+238+348+158+358+458+168+178+378+129+349+569+789
 25 h=9100105e0018c90940de7  +123+124+134+234+125+135+245+246+156+256+356+456+137+237+257+357+367+128+138+158+358+168+678+129+349+569+789
 26 h=8112543c34f0a0a825ba2  +124+134+135+345+136+146+246+356+456+257+357+267+128+138+258+358+458+168+468+178+278+139+239+149+259+359+569+379+479+189+289+789
 27 h=8100001400139e0448711  +123+135+235+136+236+246+156+356+127+137+237+147+247+257+167+238+248+358+569+789
 28 h=81014000008301015239a  +124+134+234+135+145+126+136+156+127+247+267+467+148+248+158+458+569+789
 29 h=81000084211bce050c632  +124+235+245+345+136+236+146+246+256+356+127+137+237+147+247+257+167+128+248+158+268+368+569+789
 30 h=81020084211bdf054c632  +123+134+234+146+347+567+148+348+158+458+368+129+139+149+349+159+359+169+469+289+589+689
 31 h=81020084219bdf05cc632  +124+234+235+245+345+136+236+146+246+156+256+356+127+137+237+147+247+157+257+167+267+128+238+248+158+268+368+468+569+789
 32 h=81021084619add018ce23  +123+124+234+245+345+136+236+146+246+346+256+356+456+137+237+147+157+257+267+128+138+238+248+158+268+368+468+569+789
 33 h=81021004e51ac909c849b  +123+124+134+234+245+345+126+246+346+156+256+356+456+127+137+237+147+347+157+257+248+158+168+268+468+278+569+789
 34 h=81021084659add09c84ba  +124+134+234+245+345+126+236+246+346+156+256+356+456+127+137+237+147+157+257+267+128+238+248+158+168+268+368+468+278+569+789
 35 h=8100105e20184309409e5  +123+124+134+145+345+146+256+247+238+458+678+129+239+149+459+179+479+579+189+289
 36 h=8100001000139d0408210  +234+136+236+246+127+237+147+247+257+167+238+248+358+569+789
 37 h=8100001000139d0408213  +123+124+234+136+236+246+127+237+147+247+257+167+238+248+358+569+789
 38 h=8100001000139d0408293  +123+124+134+234+136+236+246+127+237+147+247+257+167+238+248+358+569+789
 39 h=8100001400139d0408311  +123+234+135+136+236+246+356+127+237+147+247+257+167+238+248+358+569+789
 40 h=81000094011bda0508414  +134+234+136+236+146+246+347+567+158+458+368+139+249+349+159+359+169+469+389+589
 41 h=81020084211bdf050c63a  +124+125+135+235+126+246+347+567+128+248+258+458+168+268+368+129+139+149+349+159+359+169+469+289+489+589+689
 42 h=81021084659a9d09c84ba  +124+134+234+345+126+236+246+346+156+256+356+456+127+137+237+147+157+257+267+128+238+248+158+168+268+368+468+278+569+789
 43 h=8188105ce502880d4802e  +124+125+345+126+246+346+156+356+456+237+147+347+357+167+367+128+248+158+358+168+268+568+278+478+569+789
 44 h=8100205681084500144fc  +134+234+125+245+126+236+146+256+356+127+137+347+457+367+128+148+358+268+129+349+569+789
 45 h=8100001000139d0408211  +123+234+136+236+246+127+237+147+247+257+167+238+248+358+569+789
 46 h=8100001400139d0408711  +123+234+135+136+236+246+356+127+137+237+147+247+257+167+238+248+358+569+789
 47 h=810000140013de0448711  +123+135+235+245+136+236+246+156+356+127+137+237+147+247+257+167+238+248+358+569+789
 48 h=810020100013950408293  +123+124+134+234+136+236+246+127+147+247+257+457+167+238+248+358+569+789
 49 h=8100201000129500083b2  +124+134+234+135+136+236+246+127+147+257+457+128+238+248+358+569+789
 50 h=81014000008301015219a  +124+134+234+135+145+126+156+127+247+267+467+148+248+158+458+569+789
 51 h=81014000008b010952192  +124+134+234+135+145+156+256+127+247+267+467+148+248+158+458+168+569+789
 52 h=8100009400139e044860b  +123+124+235+126+136+236+246+156+356+137+237+147+247+257+167+238+248+358+368+569+789
 53 h=8100009400139e054860b  +123+124+235+126+136+236+246+156+356+137+237+147+247+257+167+238+248+158+358+368+569+789
 54 h=81000094211b4a0508c23  +123+124+235+245+345+256+356+137+237+147+247+257+167+128+138+248+158+358+268+368+569+789
 55 h=81021004e51ac909c869b  +234+135+235+136+156+157+267+567+138+238+158+258+168+368+178+578+678+149+249+159+359+169+179+279+689+789
 56 h=81021084459a9d0988623  +123+124+234+136+236+246+346+256+356+456+137+237+147+157+257+267+128+238+248+158+168+268+368+468+278+569+789
 57 h=8180100ce512cb09481b8  +134+234+135+235+245+345+126+246+346+156+356+456+127+237+147+347+257+357+128+248+158+168+268+568+278+569+789
 58 h=8188100ce512c909481b8  +134+234+135+245+345+126+246+346+156+356+456+127+237+147+347+257+357+128+248+158+168+268+568+278+478+569+789
 59 h=8100105e20180209409e5  +123+134+125+135+235+345+156+256+356+456+257+357+367+128+138+158+358+168+129+349+569+789
 60 h=81002056810845001c4fd  +123+134+234+125+245+126+236+146+256+356+127+137+147+347+457+367+128+148+358+268+129+349+569+789
 61 h=810230cb818840041c872  +124+245+146+256+456+127+147+347+357+457+167+267+367+128+138+148+348+268+368+468+129+349+569+789
 62 h=81022096810841001ace4  +134+234+125+145+245+256+356+137+147+347+457+128+138+148+358+268+368+468+129+349+569+789
 63 h=810070d3a00aca2942863  +123+124+235+145+245+345+246+156+256+456+237+347+457+367+128+138+248+348+158+358+458+168+368+178+129+349+569+789
 64 h=810070d7a00aca2942862  +124+235+145+245+345+246+156+256+356+456+237+347+457+367+128+138+248+348+158+358+458+168+368+178+129+349+569+789
 65 h=8102209a810841201ce7c  +234+125+245+126+136+146+256+127+137+147+347+357+457+128+138+148+358+268+368+468+178+129+349+569+789
 66 h=8102209b810841201ccfc  +125+135+126+136+236+146+127+347+567+128+248+158+168+568+129+239+349+469+389+489+589+689+789
 67 h=8100525720089a29420c2  +124+134+235+145+345+246+156+256+356+456+237+367+238+348+158+358+458+168+178+378+129+349+569+789
 68 h=810a304be18a18041c0d7  +123+124+134+125+345+146+346+256+456+127+237+147+347+357+457+167+267+367+238+148+248+348+268+468+478+129+349+569+789
 69 h=810000000002d00008190  +134+135+245+246+127+147+238+248+569+789
 70 h=810000940013de054860a  +124+235+245+126+136+236+246+156+356+137+237+147+247+257+167+238+248+158+358+368+569+789
 71 h=81000094001bda050861c  +125+235+245+126+136+246+256+356+127+137+237+147+247+257+167+238+248+158+358+368+569+789
 72 h=81000094011bda050841c  +134+234+136+236+146+246+347+567+158+458+368+139+249+349+159+359+169+469+389+489+589
 73 h=81801004e51a490948732  +124+234+135+245+345+136+346+156+256+356+456+127+137+237+147+347+257+128+248+158+168+268+568+278+569+789
 74 h=81801004e51a490948733  +123+124+234+135+245+345+136+346+156+256+356+456+127+137+237+147+347+257+128+248+158+168+268+568+278+569+789
 75 h=818122084091c624c4036  +124+125+235+245+236+146+246+346+156+127+247+157+257+357+457+167+267+467+128+568+178+378+569+789
 76 h=81005057a00a8b29420e4  +134+234+125+235+145+345+246+156+256+356+456+237+347+367+128+248+348+158+358+458+168+178+129+349+569+789
 77 h=810070d3a02aca2946863  +123+124+235+145+245+345+146+246+156+256+456+237+347+457+367+128+138+248+348+158+258+358+458+168+368+178+129+349+569+789
 78 h=810a304fc18a5c0418857  +124+134+125+135+145+245+126+246+256+127+347+567+258+458+168+239+149+249+349+159+259+459+169+469+189+389+589+789
 79 h=81007057a00acb29428e4  +123+124+134+135+235+145+345+126+236+346+256+137+247+347+238+458+678+239+149+249+349+459+179+479+579+679+189+289
 80 h=9100104e0018490140cf5  +123+134+234+145+345+126+146+256+567+238+458+678+129+239+149+249+459+169+179+479+189+289
 81 h=81000084211bde050c632  +124+125+135+235+126+347+567+128+148+258+458+168+268+368+139+149+349+159+359+169+469+289+589+689
 82 h=81014000008b010952992  +124+134+234+135+145+156+256+127+247+267+467+138+148+248+158+458+168+569+789
 83 h=81020084219bdf05cc631  +123+234+235+245+345+136+236+146+246+156+256+356+127+137+237+147+247+157+257+167+267+128+238+248+158+268+368+468+569+789
 84 h=81000017e008d50006f07  +123+124+234+125+135+145+245+345+136+236+146+246+346+256+356+137+347+138+238+348+358+349+569+789
 85 h=810a208780095f0106eb2  +124+134+234+235+145+245+136+236+146+256+356+127+137+237+247+347+457+128+138+238+348+158+368+468+478+349+569+789
 86 h=81c27052441b08055865e  +123+124+134+135+236+456+137+347+157+238+248+158+258+458+178+378+478+678+139+239+349+259+169+269+379+479+489
 87 h=9100104e0018490140df5  +123+134+234+145+345+126+146+256+567+238+458+678+129+239+149+459+169+179+479+189+289
 88 h=8000200c0030150046997  +123+125+245+146+346+127+147+247+678+129+139+149+349+359+459+169+269+279
 89 h=8000200c2030140046b93  +123+124+134+135+145+345+136+236+146+156+356+127+257+357+457+138+238+258+789
 90 h=810000000002d00008085  +123+134+125+245+246+147+238+248+569+789
 91 h=8100201000129504083b2  +124+134+234+135+136+236+246+127+147+257+457+167+128+238+248+358+569+789
 92 h=81000094011bda050861c  +125+235+245+126+136+246+256+356+127+137+237+147+247+257+167+238+248+158+358+268+368+569+789
 93 h=81020084619adf018ce33  +123+124+134+234+146+347+567+148+158+358+168+268+368+129+139+149+349+159+259+359+459+169+469+189+589+689
 94 h=8180100ce51ac909481a8  +134+234+135+245+345+126+246+346+156+256+356+456+237+147+347+257+357+128+248+158+168+268+568+278+569+789
 95 h=8180220900915c24c4617  +123+124+125+245+136+236+146+156+127+137+237+247+157+257+357+457+167+267+238+348+568+178+378+569+789
 96 h=8188104ce50288094802c  +125+345+126+246+346+156+356+456+237+147+347+357+367+128+248+158+168+268+568+278+478+569+789
 97 h=810010820000180810338  +135+126+136+456+127+237+128+238+148+168+368+349+569+789
 98 h=8301600340801004509a0  +134+135+346+156+457+167+267+467+128+138+238+148+348+458+578+349+569+789
 99 h=810140020401cc0800047  +123+124+125+245+236+246+237+247+467+458+168+278+129+349+569+789
100 h=810a30cbc18843041c0fa  +124+134+234+235+245+126+146+346+256+456+127+147+347+357+457+167+267+367+128+148+348+268+368+468+478+129+349+569+789
101 h=810140000083010952392  +124+134+234+135+145+136+156+127+247+267+467+148+248+158+458+168+569+789
102 h=8180200980194924c449f  +123+124+134+234+125+245+126+146+156+256+127+137+237+247+347+157+257+357+457+167+348+568+178+569+789
103 h=934140d8e0a3170c0653a  +124+234+135+235+145+345+126+236+146+346+127+137+247+347+357+167+267+367+467+567+128+238+248+258+358+458+168+368+578+678+569+789
104 h=8100205e80084800005fc  +134+125+135+245+126+256+356+127+137+237+347+357+457+367+128+358+129+349+569+789
105 h=8383004741901d2882b9e  +123+125+235+245+126+146+246+156+356+127+237+157+257+357+457+467+238+458+678+139+149+349+159+169+269+469+279+579+679
106 h=918a600ea492c40c4e5bc  +123+134+234+125+135+236+246+346+256+456+127+347+567+128+138+248+348+158+458+468+239+149+359+459+269+569+389+489+589+689
107 h=8100525720189a29401c2  +124+134+135+235+345+246+156+256+356+456+237+257+367+238+348+158+358+458+168+178+378+129+349+569+789
108 h=91000092048ac22086a41  +123+235+145+245+136+146+246+256+157+267+138+248+358+368+178+278+678+129+349+569+789
109 h=810020100012950408293  +123+124+134+234+136+236+246+127+147+257+457+167+238+248+358+569+789
110 h=810000940013de0508618  +235+245+126+136+236+246+356+127+137+237+147+247+257+167+238+248+158+358+368+569+789
111 h=81000094011bda0108423  +123+124+235+245+246+256+356+137+237+147+247+257+128+238+248+158+358+268+368+569+789
112 h=81002007e001550106f07  +123+124+234+125+135+145+245+345+136+236+146+346+356+137+247+347+457+138+238+348+158+349+569+789
113 h=8100100a00000801008dc  +134+125+126+456+127+237+357+138+158+129+349+569+789
114 h=8100205e800048000054e  +124+125+135+245+126+356+137+237+347+357+457+367+358+129+349+569+789
115 h=8102501200020f201c2de  +124+134+234+125+235+126+136+236+146+456+127+237+147+148+248+358+458+468+178+129+349+569+789
116 h=810200c6e41adf0814fc0  +125+345+246+127+137+237+347+567+138+248+348+168+368+468+568+178+378+478+249+269+469+179+479+389+489+589+689+789
117 h=810a304fc18a5c00008e4  +134+125+245+236+346+256+356+456+237+347+357+457+267+367+128+138+238+248+348+268+468+478+129+349+569+789
118 h=838330d38491130016f54  +234+125+135+235+145+136+146+456+127+137+247+347+257+457+267+367+467+138+238+148+348+358+368+468+568+278+578+129+349+569+789
119 h=91020092048ac30086a51  +123+234+235+145+245+136+146+246+256+127+157+267+138+248+358+368+468+278+678+129+349+569+789
120 h=8101408ba400c9014e92b  +123+124+234+135+145+245+345+126+146+246+156+237+147+347+357+467+128+138+348+158+458+368+278+349+569+789
121 h=810000000002d00008191  +123+134+135+245+246+127+147+238+248+569+789
122 h=810000940013de054860b  +134+234+136+236+146+246+347+567+148+248+348+158+458+368+139+149+159+359+169+189+489
123 h=81000094211b4e050cc33  +123+124+235+245+345+236+146+256+356+127+137+237+147+247+257+167+128+138+248+158+358+268+368+569+789
124 h=81021084e51ad909c84ba  +124+134+234+245+345+126+246+346+156+256+356+456+127+137+237+147+347+157+257+128+238+248+158+168+268+368+468+278+569+789
125 h=81c820094089c62086481  +123+134+235+145+245+236+146+246+346+256+137+247+157+357+457+267+567+348+568+178+478+569+789
126 h=81882049408b462080c8b  +123+124+134+235+245+126+236+346+256+137+247+157+357+457+267+367+138+248+348+568+178+478+569+789
127 h=8181104641b9de288ac9e  +124+134+125+235+145+245+126+236+246+346+256+356+456+127+137+237+147+247+157+257+267+367+467+138+238+258+168+268+568+178+349+569+789
128 h=81007057a00a8b29420e4  +134+234+125+235+145+345+246+156+256+356+456+237+347+457+367+128+248+348+158+358+458+168+178+129+349+569+789
129 h=810220db818841201cc79  +123+124+145+345+146+456+457+167+367+567+168+278+578+159+459+179+479+579+389+589
130 h=8000200c2030140046b8b  +123+124+134+135+145+345+126+136+236+146+156+356+257+357+457+138+238+258+789
131 h=8000200c0030150052987  +123+124+134+234+125+135+145+236+156+356+257+357+457+138+238+148+258+789
132 h=8001100500088c0c88097  +123+124+134+125+236+246+256+356+456+127+237+147+157+167+467+348+168+789
133 h=82005204409298008491a  +124+135+126+146+246+346+356+456+127+237+157+257+267+138+238+248+458+378+578+789
134 h=9000520c00299924c6b80  +134+234+135+145+136+146+246+156+256+356+456+237+247+157+357+167+138+238+258+458+178+378+678+789
135 h=8101204801010d08deb8b  +135+145+347+567+128+138+148+348+258+358+168+468+129+149+249+159+469+189+489
136 h=81000094011b4a0508c23  +123+124+235+245+256+356+137+237+147+247+257+167+128+138+248+158+358+268+368+569+789
137 h=8180200980184b20c049e  +124+134+234+125+235+245+126+156+256+127+137+237+347+157+257+357+457+348+568+178+569+789
138 h=834100c8e0a3570c4653f  +134+245+236+456+127+137+347+357+128+148+248+348+158+178+378+478+578+129+139+239+149+349+359+279+479+189+389+589+789
139 h=81000017c000d50004d06  +124+234+125+135+245+236+146+246+346+356+137+347+138+238+348+358+349+569+789
140 h=810000024100440004850  +245+236+146+346+127+138+268+129+349+569+789
141 h=810000d260104a08c6068  +235+145+245+345+126+146+346+156+237+157+257+367+128+358+168+368+129+349+569+789
142 h=81c0100200300c040a547  +123+124+125+135+145+236+456+137+237+147+257+167+567+258+568+129+349+569+789
143 h=8100105f20181a29408c4  +134+125+235+345+156+256+356+456+237+257+357+367+138+238+348+158+358+168+178+129+349+569+789
144 h=810220db810841201ccf6  +124+134+234+125+245+146+256+127+137+147+347+357+457+367+128+138+148+348+358+268+368+468+178+129+349+569+789
145 h=810a304fc18a1c0418057  +123+124+125+236+346+256+356+456+127+237+147+347+357+457+167+267+367+238+148+248+348+268+468+478+129+349+569+789
146 h=9100000a2088972086dec  +134+234+125+135+235+145+345+126+236+146+246+256+137+157+357+267+128+138+238+178+678+129+349+569+789
147 h=81c21016201b0804584fd  +123+134+125+345+126+156+256+356+456+127+137+237+147+247+257+167+567+128+148+248+358+468+568+129+349+569+789
148 h=810300d6c4120509d0f75  +123+125+145+345+146+456+247+257+167+267+367+567+148+158+258+458+268+468+249+169+469+569+179+679+389+489+689
149 h=8000200c0030150056997  +123+124+134+234+125+135+145+236+146+156+356+127+257+357+457+138+238+148+258+789
150 h=810020100012940008392  +124+134+135+136+236+246+127+147+257+457+238+248+358+569+789
151 h=8108104a8401cb2516781  +123+134+234+135+235+145+245+136+146+246+456+137+237+247+347+357+167+367+148+158+178+278+478+349+569+789
152 h=8108104a44010b21167a8  +134+234+135+235+145+126+136+146+346+456+137+237+247+357+367+128+148+158+178+278+478+349+569+789
153 h=8000200c0030150052996  +123+125+245+146+346+127+147+247+678+139+149+349+459+169+269+279+379
154 h=81000217a111cc081249e  +124+134+125+145+245+345+126+236+246+356+127+137+237+247+347+257+148+348+358+168+268+378+349+569+789
155 h=8001100c210045200049b  +123+124+134+234+245+345+126+236+356+456+127+137+357+467+268+178+789
156 h=9000520c00299920c6b89  +123+134+234+135+145+126+136+146+246+156+256+356+456+237+247+157+357+138+238+258+458+178+378+678+789
157 h=81405000000095010a480  +134+234+145+236+246+456+137+147+567+238+158+458+569+789
158 h=8180200000180221c408c  +134+125+235+126+146+156+256+157+257+457+158+568+178+569+789
159 h=8188104ce50288094802e  +124+125+345+126+246+346+156+356+456+237+147+347+357+367+128+248+158+168+268+568+278+478+569+789
160 h=9140624ac598d720444b2  +123+234+125+345+127+137+247+347+367+467+138+348+368+178+478+139+239+149+249+269+179+679+289+389+589+789
161 h=9301100a44328e0800d01  +123+135+235+236+246+346+456+137+237+257+357+467+138+248+258+168+278+578+678+349+569+789
162 h=8102401200020f009824b  +123+124+234+235+126+136+236+237+147+157+148+248+358+458+468+129+349+569+789
163 h=810140822401d808008d0  +134+245+345+246+127+237+247+467+138+238+458+168+368+278+129+349+569+789
164 h=83800202e421120002b68  +124+234+235+136+127+147+247+457+167+467+238+458+678+249+349+159+179+189+289
165 h=918250cf4081900886cf0  +134+145+146+246+346+356+456+127+137+247+157+357+267+367+128+138+238+348+458+168+368+468+568+678+129+349+569+789
166 h=8000300c2030140046b8b  +123+124+134+135+145+345+126+136+236+146+156+356+456+257+357+457+138+238+258+789
167 h=800020440030100012882  +124+134+145+356+257+457+367+138+238+148+258+789
168 h=8001100c21004f200049b  +123+124+134+234+235+245+345+126+236+356+456+127+137+237+357+467+268+178+789
169 h=8200729561b09b0884919  +123+234+135+235+345+126+146+246+346+356+456+127+237+157+257+457+267+138+238+348+258+358+458+168+268+368+378+578+789
170 h=9000520c00299924c6b81  +123+134+234+135+145+136+146+246+156+256+356+456+237+247+157+357+167+138+238+258+458+178+378+678+789
171 h=810020100012950408392  +124+134+234+135+136+236+246+127+147+257+457+167+238+248+358+569+789
172 h=810000020000400800190  +134+135+245+127+168+349+569+789
173 h=918a1003c0020f0c08dd1  +123+124+125+235+345+126+236+137+357+457+367+567+238+458+678+129+239+249+169+369+569+279+579+189+289
174 h=8001100500084c0484415  +123+125+245+236+146+256+356+456+127+137+237+157+167+467+348+789
175 h=818000020080000002081  +123+134+145+267+568+349+569+789
176 h=8101120200004a0002048  +235+145+245+126+456+237+467+378+129+349+569+789
177 h=8000204c0030100012982  +124+134+135+145+356+257+357+457+367+138+238+148+258+789
178 h=820052044092980004932  +124+135+146+246+346+356+456+127+237+257+267+128+138+238+248+458+378+578+789
179 h=920030456001050812e1c  +234+125+145+345+126+136+236+346+356+456+127+137+247+457+367+138+148+348+168+578+678+789
180 h=81020084211bde050c638  +235+245+345+126+136+236+146+246+256+356+127+137+237+147+247+257+167+128+238+248+158+268+368+468+569+789
181 h=81021084459a9d0988622  +124+234+136+236+246+346+256+356+456+137+237+147+157+257+267+128+238+248+158+168+268+368+468+278+569+789
182 h=83485009c520142d041ab  +123+124+134+135+126+236+146+346+456+347+357+167+567+128+238+348+158+258+458+168+268+178+278+478+578+569+789
183 h=81800043a0b198250603f  +123+124+125+145+345+126+146+246+127+237+247+347+257+167+267+367+128+238+348+158+258+568+178+349+569+789
184 h=8101400b841098004812d  +123+125+135+126+246+156+237+147+347+257+357+467+128+238+348+458+278+349+569+789
185 h=810810424081020100024  +125+235+346+456+247+267+367+128+158+478+349+569+789
186 h=8100000a01185905884d4  +134+234+125+245+256+127+137+237+147+157+257+357+167+238+158+268+129+349+569+789
187 h=8108000240085520448f3  +123+124+134+234+245+236+146+346+156+256+127+128+138+238+178+478+129+349+569+789
188 h=8001100500088c0c88013  +134+245+126+567+258+358+458+468+149+349+159+259+189+589
189 h=800310008080820002903  +123+124+135+235+145+246+456+347+267+467+138+468+789
190 h=8000100c610244200249b  +123+124+134+145+245+345+126+236+346+356+456+127+137+357+248+268+178+789
191 h=810000000002940008190  +134+135+236+246+127+147+238+248+569+789
192 h=914062026518d6204263d  +123+125+235+145+245+345+126+136+236+246+346+156+256+127+137+257+457+567+128+238+458+268+178+278+378+678+349+569+789
193 h=810000068000580000049  +123+245+126+356+237+347+238+129+349+569+789
194 h=810000140013de044871b  +123+125+236+456+127+137+347+248+378+478+239+259+279+379+479+289+389+489+589+789
195 h=814062806021950842015  +123+234+125+145+345+236+246+346+156+127+247+457+567+238+258+458+168+368+378+569+789
196 h=9301300a44328e0800900  +135+235+236+246+346+456+237+257+357+457+467+138+248+258+168+278+578+678+349+569+789
197 h=8100600a400192080425e  +124+125+235+126+136+146+246+346+127+247+357+457+238+458+168+129+349+569+789
198 h=81c80097213a480944655  +123+125+245+345+136+146+156+256+356+127+137+237+257+567+248+348+158+258+358+168+268+368+568+478+129+349+569+789
199 h=810200002011000080209  +123+345+126+136+247+157+257+468+569+789
200 h=8180200980184924c049c  +134+234+125+245+126+156+256+127+137+237+347+157+257+357+457+167+348+568+178+569+789
201 h=8180629a0580c405509c9  +123+134+135+245+126+236+246+156+357+457+167+267+138+148+158+358+458+268+368+568+278+378+129+349+569+789
202 h=820052044092980004936  +124+125+135+146+246+346+356+456+127+237+257+267+128+138+238+248+458+378+578+789
203 h=810000020000400800199  +123+134+135+245+126+127+168+349+569+789
204 h=8101420b8020000140120  +135+156+347+357+467+128+348+158+258+458+378+349+569+789
205 h=8000300c0030940046aa9  +134+135+567+128+238+148+348+168+268+149+359+169+269+369+189+289+489+689
206 h=8001100500088c0c8800f  +345+136+246+127+167+128+278+378+678+239+369+379+489
207 h=8001100500088c0c88017  +123+124+125+236+246+256+356+456+127+237+147+157+167+467+348+168+789
208 h=8000520c00299620c6aa1  +123+134+235+145+136+236+146+246+156+256+356+456+247+157+357+128+138+238+258+458+178+378+789
209 h=810000420001020100024  +125+235+247+367+128+158+349+569+789
210 h=810000028000580000049  +123+245+126+237+347+238+129+349+569+789
211 h=9000520c00299520c4b88  +134+234+135+126+136+236+146+246+156+256+356+456+247+157+357+138+238+258+458+178+378+678+789
212 h=9101024a21308f090c92d  +123+234+125+135+235+345+126+236+146+246+237+147+257+357+367+467+128+138+158+258+168+268+378+678+349+569+789
213 h=81021007400084219cfd8  +134+135+126+136+236+146+246+346+356+456+127+137+147+157+138+148+348+158+468+178+129+349+569+789
214 h=810200c6e412de0c96de3  +123+124+134+135+235+145+245+345+236+146+246+346+356+137+237+347+157+257+167+367+128+138+238+148+248+168+368+468+278+129+349+569+789
215 h=810000000002d00008110  +135+245+246+127+147+238+248+569+789
216 h=8000204c0030100012986  +124+134+125+135+145+356+257+357+457+367+138+238+148+258+789
217 h=800140408000910042b99  +123+134+234+135+145+126+136+246+156+127+347+367+467+138+238+458+789
218 h=8100200a2410470000904  +234+125+135+235+245+345+236+257+357+457+138+278+349+569+789
219 h=9143701b40010600189db  +123+124+134+135+235+126+236+346+456+127+147+247+357+457+467+567+138+148+348+358+458+468+678+129+349+569+789
220 h=914b700b000b0200189df  +123+124+134+125+135+235+126+256+456+127+147+247+357+457+467+567+138+148+248+348+458+468+478+678+129+349+569+789
221 h=810000060402440040018  +245+126+236+156+356+127+248+278+349+569+789
222 h=9180100380014d0c0c6e4  +134+234+125+245+136+236+146+456+137+237+147+247+347+167+128+348+168+568+678+129+349+569+789
223 h=91082086603047259cc6d  +123+234+125+235+245+345+126+236+146+346+356+137+147+157+257+457+167+128+138+148+158+258+368+178+478+678+129+349+569+789
224 h=810a4000a0104b0002090  +134+234+235+145+245+345+127+237+347+257+458+468+478+569+789
225 h=80c1204405191b0404422  +124+234+235+146+256+356+137+237+247+257+457+167+367+467+567+128+238+268+568+278+789
226 h=8000520c00289520c6aa0  +134+234+145+136+236+146+246+156+256+356+456+157+357+128+138+238+258+458+178+378+789
227 h=82825040a49b4729506a9  +123+134+234+235+245+345+126+136+236+156+256+456+137+247+347+257+267+367+128+148+248+158+458+168+468+568+178+278+578+789
228 h=900810c0c02947059ac24  +234+125+235+145+245+236+346+256+456+137+147+247+347+157+167+367+128+138+148+158+258+368+478+678+789
229 h=830012124080980412f5a  +124+135+145+126+136+246+346+456+127+137+237+167+267+138+238+148+358+378+578+129+349+569+789
230 h=83c2308cc090862dc0f30  +124+125+135+245+345+236+146+346+156+256+456+257+357+457+467+458+678+249+259+459+169+469+569+179+479+579+679
231 h=8000300c2030940046aab  +123+124+134+145+345+126+136+236+146+246+156+356+456+257+357+457+128+138+238+258+789
232 h=8001100500088c0c8802f  +123+124+125+126+236+246+256+356+456+237+147+157+167+467+128+348+168+789
233 h=9000520c00299420c4b80  +134+135+136+236+146+246+156+256+356+456+247+157+357+138+238+258+458+178+378+678+789
234 h=8100105f20190a29408c7  +123+124+134+125+235+345+156+256+356+456+237+247+257+357+367+138+348+158+358+168+178+129+349+569+789
235 h=8000000ca00041004cf83  +123+124+134+234+135+245+345+136+146+156+356+137+147+347+357+138+789
236 h=828210408493422840e88  +134+235+245+126+136+156+456+137+247+347+257+267+367+138+248+168+468+568+178+278+578+789
237 h=800040004110000400300  +135+136+346+257+167+458+268+789
238 h=810000000002d00008112  +124+135+245+246+127+147+238+248+569+789
239 h=810010020510490080200  +234+245+136+456+237+157+257+268+278+349+569+789
240 h=8000200c0030100012984  +134+125+135+145+356+257+357+457+138+238+148+258+789
241 h=8040201145b1990504013  +123+124+234+146+246+346+127+237+247+257+457+167+267+567+238+348+158+258+358+268+278+789
242 h=800140800400910044b3e  +124+234+125+135+126+136+146+246+156+127+467+128+138+238+458+368+278+789
243 h=80090241858900001c814  +125+146+256+127+147+247+347+267+367+467+138+148+348+268+278+378+478+789
244 h=9200001181b000288c990  +134+135+146+127+147+347+157+257+267+138+348+258+358+168+268+178+578+678+789
245 h=810100020010040000403  +123+124+236+137+257+467+349+569+789
246 h=800110000000440080086  +124+134+125+245+236+456+157+467+789
247 h=9000520c00299420c4a80  +134+136+236+146+246+156+256+356+456+247+157+357+138+238+258+458+178+378+678+789
248 h=810000000002000008003  +123+124+147+248+569+789
249 h=810a100220004000081e2  +124+134+135+245+345+456+147+128+468+478+129+349+569+789
250 h=8308128a000a5e20c6a80  +134+235+145+245+136+236+146+156+256+456+237+157+357+138+238+248+368+178+378+478+578+349+569+789
251 h=8000520c00299720c6aa0  +134+234+235+145+136+236+146+246+156+256+356+456+247+157+357+128+138+238+258+458+178+378+789
252 h=8008328040239901da601  +123+234+145+136+246+346+156+456+137+237+147+247+157+457+238+148+248+158+258+368+378+478+789
253 h=8202000020001b0c8a097  +123+124+134+234+125+235+145+345+127+237+147+157+167+238+168+468+578+789
254 h=900a0201a432170000c22  +124+234+235+345+236+137+347+257+128+138+238+248+348+258+468+278+378+478+678+789
255 h=810000000002940008192  +124+134+135+236+246+127+147+238+248+569+789
256 h=8100200000100001c4081  +123+134+146+156+157+257+457+158+569+789
257 h=83820008a5814e0880b2c  +124+134+125+126+136+246+156+256+157+257+357+457+167+267+467+458+678+149+159+469+179+579
258 h=8240728d05b08e08d498d  +123+134+125+135+235+126+236+146+246+156+356+456+237+157+257+357+457+267+567+138+148+348+258+458+168+268+368+278+378+578+789
259 h=800110000000440080083  +123+124+134+245+236+456+157+467+789
260 h=810860420182800118eea  +124+134+126+136+246+137+147+457+267+367+128+138+148+248+158+458+268+478+129+349+569+789
261 h=800000400030000012892  +124+134+145+127+257+367+138+148+258+789
262 h=800140800400910040b3e  +124+234+125+135+126+136+246+156+127+467+128+138+238+458+368+278+789
263 h=800140808400910044b3c  +234+125+135+126+136+146+246+156+127+347+467+128+138+238+458+368+278+789
264 h=80090249c5a109001c213  +123+124+234+136+146+346+127+237+147+247+347+357+267+367+467+148+348+258+268+278+378+478+789
265 h=800110000000440080087  +123+124+134+125+245+236+456+157+467+789
266 h=80c0009c808104080040d  +123+125+126+236+356+137+247+347+357+267+567+358+168+368+568+789
267 h=828050000080402950321  +123+135+245+136+156+456+267+128+148+158+458+168+568+178+578+789
268 h=810000000000800002412  +124+145+246+127+137+569+789
269 h=818020084091c724c441e  +124+234+125+235+245+126+236+146+246+346+156+127+137+247+157+257+357+457+167+267+568+178+569+789
270 h=8100000a2410470000904  +234+125+135+235+245+345+236+257+357+138+278+349+569+789
271 h=8041100000004f0480212  +124+234+235+245+136+236+456+127+237+157+167+467+567+789
272 h=800020400030100012882  +124+134+145+257+457+367+138+238+148+258+789
273 h=810000122001040000040  +345+236+247+358+129+349+569+789
274 h=81c21092618b0d2082e69  +123+234+145+345+126+136+236+346+256+456+137+237+247+157+267+567+128+138+248+358+268+368+468+568+178+129+349+569+789
275 h=810000420000020000024  +125+235+367+128+349+569+789
276 h=8180200000100021c4099  +123+134+126+146+156+127+157+257+457+158+568+178+569+789
277 h=8000200c0030110052986  +124+134+234+125+135+145+156+356+257+357+457+138+238+148+258+789
278 h=0240008020000d000218c  +134+234+125+135+145+345+126+236+237+567+368+578
279 h=800000400030000010892  +124+134+127+257+367+138+148+258+789
280 h=810000420000830000011  +123+234+235+246+127+367+349+569+789
281 h=810000020000500000048  +245+126+238+129+349+569+789
282 h=810000026002000002404  +125+145+345+346+137+248+349+569+789
283 h=800000400030000010812  +124+127+257+367+138+148+258+789
284 h=810000000002100008003  +123+124+147+238+248+569+789
285 h=0000201400209b054e081  +123+134+234+235+145+146+246+156+356+237+147+457+167+238+158+258+358
286 h=800000400010000010817  +123+124+125+127+257+367+138+148+789
287 h=810000028000000004340  +135+136+146+347+129+349+569+789
288 h=810100020001000800042  +124+247+467+168+129+349+569+789
289 h=00002010408096054a586  +124+125+135+346+156+256+127+247+139+149+349+459+269+369+569+479
290 h=81000002a000520010274  +125+235+245+345+136+127+347+128+238+148+129+349+569+789
291 h=81001000200000200041f  +123+124+125+345+126+456+127+137+178+569+789
292 h=918212030001ce095231c  +124+234+125+345+126+146+167+367+348+368+178+378+129+249+169+269+469+179+379+189+389+489+589
293 h=800000050000400000001  +123+245+356+348+789
294 h=818000028000100010480  +134+137+347+238+148+568+349+569+789
295 h=024000802000840082084  +134+125+145+345+236+246+157+567+368+578
296 h=810100020000000800042  +124+467+168+129+349+569+789
297 h=808b7000a008cf245ab3b  +123+124+234+135+235+145+245+345+126+136+236+246+156+256+456+127+237+147+347+457+167+467+128+138+148+458+468+568+178+478+789
298 h=02400080200084008208c  +134+125+145+345+126+236+246+157+567+368+578
299 h=9048000884a21c20c4819  +123+126+236+146+156+127+237+347+157+357+267+567+138+238+248+258+178+278+478+678+789
300 h=810000000002000000003  +347+567+269+189
301 h=f5194b66db1322ba5bd0d  +123+125+135+235+145+126+346+156+356+137+147+247+347+257+367+467+138+148+248+458+168+268+178+378+478+678+139+239+349+159+359+169+269+369+569+279+479+579+679+189+289+389+689+789
302 h=0000204c4008430050016  +124+234+125+235+245+346+156+256+356+127+357+457+367+148
303 h=0240008020008d000218c  +134+234+125+135+145+345+126+236+246+237+567+368+578
304 h=100000004010010002297  +123+124+134+234+125+145+136+346+127+257+678
305 h=800000400010000010812  +124+127+257+367+138+148+789
306 h=808000002100410002c16  +124+234+125+145+245+345+127+137+138+268+568+789
307 h=810000060409160000451  +123+235+236+256+356+127+137+247+238+278+129+349+569+789
308 h=020860c1a030dd2906c0a  +124+234+145+245+345+126+236+146+246+137+237+347+257+457+367+138+238+348+158+258+458+168+368+178+478+578
309 h=8182000a0000400108000  +245+147+357+158+468+568+349+569+789
310 h=810000024000100000441  +123+346+137+238+129+349+569+789
311 h=800000500131000080421  +123+137+247+157+257+367+128+258+358+268+789
312 h=12400080200085000218c  +123+124+134+167+248+178+249+189+289+689+789
313 h=810000020000400010002  +124+245+148+349+569+789
314 h=00002001041098000448e  +124+134+125+126+146+246+137+237+257+457+238+348+278
315 h=800010000000000000202  +124+136+456+789
316 h=0000201800a093054e480  +134+234+235+145+146+246+156+137+147+357+457+167+267+238+158+258+358
317 h=8100005600000e0000001  +123+235+236+356+237+367+358+349+569+789
318 h=00002018008093054e4a4  +134+234+125+235+145+146+246+156+137+147+357+457+167+267+128+238+158+358
319 h=810000020000580000040  +245+237+238+129+349+569+789
320 h=800000040000440040380  +134+135+245+136+236+156+356+789
321 h=024000000000840082084  +134+125+145+236+246+157+567+578
322 h=810010020000400010002  +124+245+456+148+349+569+789
323 h=800000100131000080421  +123+137+247+157+257+128+258+358+268+789
324 h=0000400080008004c8e26  +124+125+136+246+156+137+147+347+157+167+128+138+458
325 h=8100000a0000060000841  +123+235+236+357+138+129+349+569+789
326 h=800000040000042000006  +124+125+236+356+178+789
327 h=810000020000400010000  +245+148+349+569+789
328 h=800140000001500008012  +124+245+127+147+247+467+238+458+789
329 h=810010210510001008001  +123+456+147+257+348+268+278+359+169+569+789
330 h=810000020000000000004  +125+349+569+789
331 h=800000000010000000817  +123+124+125+127+257+138+789
332 h=000010008100000410000  +456+347+167+148+268
333 h=0000100020024300d0008  +234+235+245+345+126+156+456+157+148+248
334 h=810000000000000000002  +124+569+789
335 h=810000020000080000000  +237+349+569+789
336 h=800000000010000010817  +123+124+125+127+257+138+148+789
337 h=810000020000020000040  +235+129+349+569+789
338 h=004000040000c20000294  +134+125+235+245+136+246+356+127+567
339 h=800000000010000000002  +124+257+789
340 h=00000000200897000020b  +123+124+234+235+345+126+136+236+246+256+238
341 h=0000204c4008c30000017  +123+124+234+125+235+245+246+346+256+356+127+357+457+367
342 h=000000000000020082014  +125+235+145+127+157
343 h=000000000000010080002  +124+234+157
344 h=000000002000080082002  +124+145+345+237+157
345 h=800100408080000000000  +347+267+367+467+789
346 h=810000020000000000040  +129+349+569+789
347 h=800000000010000000000  +257+789
348 h=000000000000000000002  +124
349 h=000000000000000000000
"""

if __name__ == "__main__":
    main()


CPU verification with 2 worker(s). Results: rm19_verification.json
1/5: catalog checksum, witness, transforms, and integer inequalities.
The first call also compiles the Numba functions.
Witness and arithmetic checks passed.
2/5: preparing both sets of quadratic representatives.
First preparation complete.
Both preparations: 193 classes, 251,133,952 representatives each.
3/5: auditing all 349 cubic forms.
Catalog audit: 50/349
Catalog audit: 100/349
Catalog audit: 150/349
Catalog audit: 200/349
Catalog audit: 250/349
Catalog audit: 300/349
Catalog: 349 distinct invariants, 32 restriction labels.
4/5: first complete enumeration.
Run 1: compiling integer loops on first use.
Run 1, class 1 (1/193): 262,144 tested, 37,202 retained, best 236
Run 1, class 3 (2/193): 262,144 tested, 35,119 retained, best 236
Run 1, class 4 (3/193): 262,144 tested, 38,867 retained, best 236
Run 1, class 5 (4/193): 262,144 tested, 41,776 retained, best 236
Run 1, class 6 (5/193): 262,144 tested, 43,265 retained